# DBSCAN Clustering Analysis

This notebook performs DBSCAN clustering on the fire dataset to identify density-based clusters and outliers.
We will use the balanced dataset for this analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Libraries imported successfully!")

In [ ]:
# Load the dataset
dataset_path = '../results/balanced_dataset.csv'
df = pd.read_csv(dataset_path)
print(f"Loaded dataset from {dataset_path} with shape {df.shape}")

# Prepare data for clustering
# We drop the target 'class' as clustering is unsupervised
# We also drop 'longitude' and 'latitude' to focus on environmental features, 
# or we can keep them if we want spatial clustering. 
# Let's drop them to find clusters based on features (climate, soil, elevation).
X = df.drop(['longitude', 'latitude', 'class'], axis=1)

# Drop non-numeric columns
X = X.select_dtypes(include=[np.number])

# Handle missing values
if X.isnull().sum().sum() > 0:
    X = X.fillna(X.mean())

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Data scaled and ready for clustering.")

In [ ]:
# Run DBSCAN
# eps: The maximum distance between two samples for one to be considered as in the neighborhood of the other.
# min_samples: The number of samples (or total weight) in a neighborhood for a point to be considered as a core point.

# These parameters might need tuning. 
# A common heuristic for min_samples is 2 * dimensions.
min_samples = 2 * X.shape[1]
eps = 3.0 # Starting guess, can be tuned

print(f"Running DBSCAN with eps={eps}, min_samples={min_samples}...")
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
labels = dbscan.fit_predict(X_scaled)

# Add labels to dataframe
df['cluster'] = labels

# Statistics
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

print(f'Estimated number of clusters: {n_clusters_}')
print(f'Estimated number of noise points: {n_noise_}')
print(f'Percentage of noise points: {n_noise_ / len(labels) * 100:.2f}%')

In [ ]:
# Visualize Clusters using PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', s=50, alpha=0.6)
plt.title('DBSCAN Clustering Results (PCA Projection)', fontsize=16)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(scatter, label='Cluster Label')
plt.show()

In [ ]:
# Evaluate Clustering
if n_clusters_ > 1:
    sil_score = silhouette_score(X_scaled, labels)
    print(f"Silhouette Score: {sil_score:.3f}")
else:
    print("Silhouette Score cannot be calculated (less than 2 clusters found).")

# Analyze Cluster Composition (Optional)
# Check how many Fire/No Fire points are in each cluster
if 'class' in df.columns:
    print("\nCluster Composition by Class (0=No Fire, 1=Fire):")
    print(pd.crosstab(df['cluster'], df['class']))